# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore a FAIR²-format dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL describing all entities, metadata, and data resources for programmatic access and reproducibility.

In [ ]:
# Ensure `mlcroissant` and plotting libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
ds = mlc.Dataset(croissant_url)
meta = ds.metadata

print(f"{meta.name}: {meta.description}\n")
print(f"Version: {meta.version}. License: {meta.license}\n")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In Croissant, each record set is uniquely identified by its `@id`. We'll inspect the dataset objects, listing all record sets and their contained fields.

In [ ]:
# List all record sets and their field @ids
record_sets = list(ds.record_sets.keys())
print("Available record sets and their field @ids:")
for recset_id in record_sets:
    recset = ds.record_sets[recset_id]
    print(f"- Record set @id: {recset_id}  ({recset.name})")
    if hasattr(recset, 'fields') and recset.fields:
        print("  Fields:")
        for field in recset.fields:
            print(f"    - {field['@id']} ({field.get('name', 'unnamed')})")
    elif hasattr(recset, 'columns') and recset.columns:
        # Some record sets use 'columns' instead of 'fields'
        print("  Columns:")
        for col in recset.columns:
            print(f"    - {col['@id']} ({col.get('name', 'unnamed')})")
    else:
        print("  (No explicit fields/columns listed)")
    print()

In [ ]:
# Show a few sample records from the primary data table (using its `@id`)
main_recset_id = record_sets[0] if record_sets else None
if main_recset_id:
    print(f"First 2 records from record set '{main_recset_id}':")
    for i, record in enumerate(ds.records(record_set=main_recset_id)):
        if i >= 2:
            break
        print(record)
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction

Load data from each record set into DataFrames, referencing record set and field `@id`s. The DataFrames enable further processing and analysis.

In [ ]:
# Load all record sets into pandas DataFrames using their @ids
dfs = {}
for recset_id in record_sets:
    data = list(ds.records(record_set=recset_id))
    df = pd.DataFrame(data)
    dfs[recset_id] = df
    print(f"Loaded record set: {recset_id}. Number of records: {len(df)}")

# Show available columns in the main record set
if main_recset_id:
    print(f"\nColumns in record set {main_recset_id}:")
    print(dfs[main_recset_id].columns.tolist())
    # Show a preview
    dfs[main_recset_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll apply common processing steps, such as filtering records by a chosen numeric field, normalizing, and grouping. All columns and groupings are referenced by their `@id`.

In [ ]:
# Select a numeric field (edit as appropriate for your dataset)
df = dfs[main_recset_id]
# Heuristic: select a numeric field (e.g., Age) by looking for typical keywords or numeric dtype
numeric_candidates = [c for c in df.columns if 'age' in c.lower() or df[c].dtype.kind in 'fi']
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # Fallback: pick the first column
    numeric_field = df.columns[0]
print(f"Selected numeric field: {numeric_field}")

# Filter records where the numeric field exceeds a threshold (e.g., Age > 50)
try:
    threshold = float(df[numeric_field].mean()) if df[numeric_field].dtype.kind in 'fi' else 50
except Exception:
    threshold = 50
filtered_df = df[df[numeric_field].astype('float', errors='ignore') > threshold].copy()
print(f"\nFiltered records with {numeric_field} > {threshold:.1f}:")
print(filtered_df.head())

# Normalize field
if filtered_df[numeric_field].dtype.kind not in 'fi':
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical/group field (e.g., 'sex', 'msi', etc.)
group_candidates = [c for c in df.columns if c.lower() in ['sex', 'gender', 'msi', 'site', 'anatomical_location'] or df[c].dtype=='object']
if group_candidates:
    group_field = group_candidates[0]
    print(f"\nGrouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("\nNo suitable categorical group field found for grouping.")

## 5. Visualization

Visualize numeric field distribution and grouped summary statistics.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouped summary is available, plot
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(8,4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} Mean by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.show()

## 6. Conclusion

- This notebook demonstrates loading, exploring, and analyzing the FAIR² clinical oncology dataset using the `mlcroissant` library.
- All metadata, fields, and records are referenced by their Croissant `@id`.
- You can use this template as a starting point to perform more detailed exploratory, statistical, or machine learning analyses on the dataset.

Explore further by examining additional fields, record sets, and relationships, or by integrating the data into your own workflows!